In [1]:
import os
import sys
import numpy as np
import pandas as pd

from datetime import datetime
from dateutil import relativedelta

import random
random.seed(123)

import plotly.graph_objs as go
import plotly.offline as pyo
import plotly.subplots as psub

# import functions
sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

from load_data import load_data
from load_spec import load_spec
from summarize import summarize

In [2]:
# pd.set_option('display.max_columns', 30)

In [3]:
## Load data
country = 'US';         # United States macroeconomic data
sample_start = datetime.strptime('2000-01-01', '%Y-%m-%d'); # estimation sample

## Load model specification and dataset.
# Load model specification structure `Spec`
Spec = load_spec('../data/0_source/Spec_US_example.xls');
# Parse `Spec`
SeriesID, SeriesName, Units, UnitsTransformed, Frequency = Spec['seriesid'], Spec['seriesname'], Spec['units'], Spec['unitstransformed'], Spec['frequency']

# Prepare data -----------------------------------------------------------
datafile = pd.read_excel("../data/02_intermediate/harmonized_time_series.xlsx", header=None)
X, Time, Z, header = load_data(datafile, Spec, sample_start);

/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/load_spec.py:41: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Table 1: Model specification
              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction Spen

In [4]:
X_df = pd.DataFrame(X, columns=header, index=Time)
Z_df = pd.DataFrame(data=Z, columns=header, index=Time)

fig = psub.make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                            subplot_titles=("Raw Observed Data", "Transformed data"))

## Plot raw and transformed data.
# Industrial Production (INDPRO) <fred.stlouisfed.org/series/INDPRO>
series_name = "GDPC1"
idxSeries = SeriesID.index(series_name)
# Plot raw observed data
trace1 = go.Scatter(
    x=Z_df.sort_index()[series_name].dropna().pct_change(4).index,
    y=Z_df.sort_index()[series_name].dropna().pct_change(4),
    mode="lines+markers" if Z_df[series_name].isna().sum() else "lines",
    name='Raw Observed Data',
    line=dict(color="#000000", width=1),  # #7BCC62 / #68b562 / #7BB562
    marker={"size": 4, "symbol": "diamond"},
)
fig.add_trace(trace1, row=1, col=1)

# Plot transformed data
trace2 = go.Scatter(
    x=X_df.sort_index().dropna().index,
    y=X_df[series_name].sort_index().dropna(),
    mode='lines',
    name='Transformed Data',
    line=dict(color="#BDC1D6", width=1),  # #7BCC62 / #68b562 / #7BB562
)
fig.add_trace(trace2, row=2, col=1)
fig.update_layout(
    height=600,
    width=800,
    showlegend=False,
    title_text=series_name,
    plot_bgcolor="white",
)
fig.update_xaxes(range=[Time[0], Time[-1]], row=1, col=1, gridcolor="lightgrey")
fig.update_yaxes(
    # title_text=Units[idxSeries],
    title_text="Percentage change Year-over-Year",
    row=1,
    col=1,
    gridcolor="lightgrey"
    )

fig.update_xaxes(range=[Time[0], Time[-1]], title_text='Time', row=2, col=1, gridcolor="lightgrey")
fig.update_yaxes(title_text=UnitsTransformed[idxSeries], row=2, col=1, gridcolor="lightgrey")
fig.show()

In [5]:
# print("Transformed monthly series:")
# display(df_m.tail(5))
# print("Transformed quarterly series:")
# display(df_q.tail(5))
# print("Raw observed data:")
# display(Z_df.tail(5))

# # target reference dates: 
# # "20XX-01-01", "20XX-04-01", "20XX-07-01", "20XX-10-01"
# reference_date = df_q[series_name].index.max() # + relativedelta.relativedelta(months=3)
# date_ranges[1] = reference_date
# print(f"Reference date: {reference_date}, estimation date ranges: {date_ranges}")

In [6]:
# df_mod_m = pd.merge(df_q, df_m, how="left", left_index=True, right_index=True)

# X_df, y = df_mod_m.drop(columns=[series_name]), df_mod_m[series_name]
# X, _, _ = remNaNs_spline(X_df.values, options={"method": 1, "k": 3})
# X = pd.DataFrame(X, columns=X_df.columns, index=X_df.index)
# # train-test split
# split_dt = df_q[series_name].index.max() - relativedelta.relativedelta(months=2)

# train_index, test_index = y.loc[y.index < split_dt].index, y.loc[(y.index >= split_dt) & (y.index <= reference_date)].index
# X_train, y_train = X.loc[X.index < split_dt],  y.loc[y.index < split_dt]
# X_test, y_test = X.loc[(X.index >= split_dt) & (X.index <= reference_date)],  y.loc[(y.index >= split_dt) & (y.index <= reference_date)]

# print("X train:")
# display(X_train)
# print("y train:")
# display(y_train)

# print("X test:")
# display(X_test)
# print("y test:")
# display(y_test)

### Growth rates retransformation

1. $ growth\_rate = (present / past) ** (1 / n) - 1 $
2. $ present = past * (1 + growth\_rate) ** n $

In [7]:
# growth_rate = y_test.resample("QS").last()
# past = Z_df[series_name].loc[y_train.resample("QS").last().index].tail(1)
# present = Z_df[series_name].loc[y_test.resample("QS").last().index].tail(1)

# step = 3
# n = step / 12

In [8]:
# # Actual growth rates calculation
# past = Z_df[series_name].loc[y_train.resample("QS").last().index].shift(1)
# present = Z_df[series_name].loc[y_train.resample("QS").last().index]

# actual_growth_rates = (present / past).dropna() ** (1 / n) - 1
# print(actual_growth_rates)

In [9]:
# # Retransformation
# # present = past * (1 + growth_rate) ** n

# # Actual present values
# print(present)  # present values from source data
# print(past * (1 + actual_growth_rates) ** n)  # present values – estimated based on raw growth rated

In [10]:
# # Smoothed growth rates and present values
# estimated_growth_rates = y_train.resample("QS").last()/100
# past * (1 + estimated_growth_rates) ** n  #  present values – estimated based on smoothed growth rated

In [11]:
# # Corrected estimates of present values (residuals added to growth rates)
# growth_rate_residuals = (actual_growth_rates - estimated_growth_rates)
# residuals_mean, residuals_std = growth_rate_residuals.mean(), growth_rate_residuals.std()
# print(residuals_mean, residuals_std)
# present_corrected1 = past * (1 + (estimated_growth_rates + np.random.normal(residuals_mean, residuals_std, len(estimated_growth_rates)))) ** n

# print(((present_corrected1 - present).abs()/present).dropna().mean())
# present_corrected1

In [12]:
# # Corrected estimates of present values (residuals added to present values)
# pred_residuals = (past * (1 + estimated_growth_rates) ** n) - present
# residuals_mean, residuals_std = pred_residuals.mean(), pred_residuals.std()
# print(residuals_mean, residuals_std)

# present_corrected2 = (past * (1 + estimated_growth_rates) ** n) + np.random.normal(residuals_mean, residuals_std, len(estimated_growth_rates))
# print(((present_corrected2 - present).abs()/present).dropna().mean())
# present_corrected2

In [13]:
# print((((present_corrected1 + present_corrected2) / 2 - present).abs()/present).dropna().mean())
# print((present_corrected1 + present_corrected2) / 2)

In [15]:
df = pd.read_csv("../data/0_source/00_extract_vintagedata.csv")

/var/folders/3k/vh6dl_9j30z3n567nqndm7tw0000gp/T/ipykernel_67219/2133498337.py:1: DtypeWarning:

Columns (2,8,9,10,11,15) have mixed types. Specify dtype option on import or set low_memory=False.



In [18]:
df.loc[(df["VariableCode"] == "GDPC1") & (df["ReferenceDate"] == "2022-10-01")]

,VariableCode,Description,Category,Region,Unit,Adjustment,FrequencyDescription,LastUpdatedOnSource,ReleaseName,ReleaseLink,SourceName,SourceLink,VariableId,ReferenceDate,PublicationDate,VariableValue
8081078,GDPC1,Real Gross Domestic Product,GDP/GNP,United States,Billions of Chained 2017 Dollars,Seasonally Adjusted Annual Rate,Quarterly,2024-09-26 00:00:00.0000000 +00:00,NaN,NaN,NaN,NaN,170500,2022-10-01,2023-01-26,"20198,091"
8081079,GDPC1,Real Gross Domestic Product,GDP/GNP,United States,Billions of Chained 2017 Dollars,Seasonally Adjusted Annual Rate,Quarterly,2024-09-26 00:00:00.0000000 +00:00,NaN,NaN,NaN,NaN,170500,2022-10-01,2024-07-25,"21989,981"
8081080,GDPC1,Real Gross Domestic Product,GDP/GNP,United States,Billions of Chained 2017 Dollars,Seasonally Adjusted Annual Rate,Quarterly,2024-09-26 00:00:00.0000000 +00:00,NaN,NaN,NaN,NaN,170500,2022-10-01,2024-09-26,"22249,459"
